In [8]:
import pandas as pd
from third_party import nobipy
from third_party.nobipy.api_client import APIClient,nobitex_client_factory
import datetime as dt
api_client = APIClient(
    client=nobitex_client_factory()
)

In [37]:
from third_party.nobipy.models import GetMarketHistoryRequest

history = api_client.get_market_history(
    GetMarketHistoryRequest(
        symbol = 'DAIUSDT',
        resolution = nobipy.models.Resolutions.D1.value,
        to = int(dt.datetime.now(dt.UTC).timestamp()),
        page = 1,
        countback=1000
    )
)

In [38]:
from dataclasses import asdict

In [39]:
d = asdict(history)
d.pop('s')

'ok'

In [255]:
df = pd.DataFrame(d)
df = df.rename(
    columns = {
        't': 'datetime',
        'o' : 'open',
        'c' : 'close',
        'h' : 'high',
        'l' : 'low',
        'v' : 'volume',
    }
)

In [256]:
import talib

df['tr'] = talib.TRANGE(df['high'], df['low'], df['close'])
df = df.dropna(subset='tr')

In [257]:
df['tr_int'] = df['tr']*10000

In [258]:
# df['kama_low'] = talib.MA(df['low'], timeperiod = 16)
df['kama_high'] = talib.SMA(df['high'], timeperiod = 9)

In [259]:
df = df.dropna(subset=['kama_high'])

In [260]:
df['kama_high'] = df['kama_high'] - 0.0001

In [261]:
df['high_touch_kama'] = df.apply(lambda x:1 if x['high'] >= x['kama_high'] else 0, axis=1)

In [262]:
df['high_touch_kama'].value_counts()

high_touch_kama
0    260
1    216
Name: count, dtype: int64

In [263]:
df[['high', 'kama_high','high_touch_kama']]

,high,kama_high,high_touch_kama
9,1.0016,1.004489,0
10,1.0010,1.003489,0
11,1.0008,1.002500,0
12,1.0040,1.002500,1
13,1.0022,1.002300,0
...,...,...,...
480,0.9996,0.998022,1
481,0.9996,0.998256,1
482,0.9990,0.998178,1
483,0.9990,0.998122,1
